In [10]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import time

df = pd.read_csv("premier_league_stats.csv")
missing_age_df = df[df['age'].isna()]

print(f"Found {len(missing_age_df)} players with missing ages")

driver = webdriver.Firefox()
wait = WebDriverWait(driver, 10)

cookie_accepted = False

try:
    for idx, row in missing_age_df.iterrows():
        player_name = row['player']
        print(f"\nSearching for: {player_name}")
        
        driver.get("https://www.transfermarkt.com")
        time.sleep(3)
        
        if not cookie_accepted:
            try:
                accept_button = wait.until(EC.element_to_be_clickable((By.XPATH, "//button[contains(text(), 'Accept') or contains(text(), 'Agree')]")))
                accept_button.click()
                cookie_accepted = True
                time.sleep(2)
            except:
                pass
        
        search_box = wait.until(EC.presence_of_element_located((By.NAME, "query")))
        search_box.clear()
        search_box.send_keys(player_name)
        
        search_button = wait.until(EC.element_to_be_clickable((By.CLASS_NAME, "tm-header__input--search-send")))
        search_button.click()
        time.sleep(3)
        
        try:
            age_element = wait.until(EC.presence_of_element_located((By.XPATH, "//table[@class='items']//tbody//tr[1]//td[@class='zentriert'][3]")))
            age = age_element.text.strip()
            
            print(f"Found age: {age}")
            df.at[idx, 'age'] = age
            
        except Exception as e:
            print(f"Could not find age for {player_name}: {e}")
        
        time.sleep(3)
    
    df.to_csv("premier_league_stats_updated.csv", index=False)
    print("\nUpdated CSV saved!")
    
finally:
    driver.quit()

print("\nDone!")

Found 5 players with missing ages

Searching for: Mateus Mane
Could not find age for Mateus Mane: Message: 
Stacktrace:
RemoteError@chrome://remote/content/shared/RemoteError.sys.mjs:8:8
WebDriverError@chrome://remote/content/shared/webdriver/Errors.sys.mjs:202:5
NoSuchElementError@chrome://remote/content/shared/webdriver/Errors.sys.mjs:555:5
dom.find/</<@chrome://remote/content/shared/DOM.sys.mjs:136:16


Searching for: Jake Evans
Found age: 17


/tmp/ipykernel_65492/3643549249.py:49: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '17' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.at[idx, 'age'] = age



Searching for: Olabade Aluko
Found age: 18

Searching for: Tom Taylor
Found age: 20

Searching for: Jayden Moore
Found age: 18

Updated CSV saved!

Done!
